In [29]:
#importuri

import numpy as np 
import pandas as pd
import os
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import torch
import torchvision
import torch.nn as nn 
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import torch.optim
from PIL import Image

In [30]:
#variabile globale si fisiere

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

tr_df,val_df = train_test_split(train_df,
                                test_size = 0.2,
                                stratify = train_df['Label'],
                                random_state = 42)
BATCH_SIZE = 32
EPOCHS = 20 
LR = 3e-4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATAS = '.'

In [31]:
#transforms

train_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
          std=[0.229,0.224,0.225]),
])

test_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485,0.456,0.406],
          std=[0.229,0.224,0.225]),
])

In [32]:
#dataset custom 

class date_arte(Dataset):
    def __init__(self,df,root_dir,transforms = None, test = False):
        self.root_dir = root_dir
        self.transforms = transforms
        self.test = test

        self.X_path = df['ImagePath'].values
        if not test:
            self.labels = df['Label'].values
        else: 
            self.labels = None

    def __len__(self):
        return len(self.X_path)
    
    def __getitem__(self,idx):
        img_path = os.path.join(self.root_dir, self.X_path[idx])
        image = Image.open(img_path).convert('RGB')

        if self.transforms:
            image = self.transforms(image)

        if self.test:
            return image, self.X_path[idx]
        else:
            label = self.labels[idx]
            return image, torch.tensor(label, dtype = torch.float32)

In [33]:
train_dts = date_arte(tr_df, DATAS, train_tfms, test = False)
val_dts = date_arte(val_df, DATAS, test_tfms, test = False)
test_dts = date_arte(test_df, DATAS, test_tfms, test = True)

train_loader = DataLoader(train_dts, batch_size = BATCH_SIZE, shuffle = True)
val_loader = DataLoader(val_dts, batch_size = BATCH_SIZE, shuffle = False)
test_loader = DataLoader(test_dts, batch_size= BATCH_SIZE, shuffle = False)

In [34]:
class ArtCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = models.resnet18(pretrained = True)
        self.model.fc = nn.Linear(self.model.fc.in_features,1)

    def forward(self,x):
        return self.model(x)

In [35]:
model = ArtCNN().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = LR)

c:\Users\sabin\Desktop\probleme_competitii_ml\venv_torch\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\sabin\Desktop\probleme_competitii_ml\venv_torch\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [36]:
def evaluate_model(model,loader,threshold = 0.5):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            outputs = model(images)
            probs= torch.sigmoid(outputs)
            preds = (probs>threshold).int().cpu().numpy().flatten()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    
    return f1_score(all_labels,all_preds)

In [37]:
best_f1 = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for images, labels in tqdm(train_loader):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE).unsqueeze(1)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    val_f1 = evaluate_model(model,val_loader)

    print(f'Epoch {epoch+1}/ {EPOCHS}') 
    print(f'Loss: {total_loss:.4f} | Val F1: {val_f1}')

    if val_f1 > best_f1:
        best_f1 = val_f1

  5%|▌         | 1/20 [00:01<00:23,  1.25s/it]c:\Users\sabin\Desktop\probleme_competitii_ml\venv_torch\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 5/5 [00:04<00:00,  1.17it/s]


Epoch 1/ 20
Loss: 10.8926 | Val F1: 0.7692307692307693


 20%|██        | 4/20 [00:05<00:20,  1.30s/it]c:\Users\sabin\Desktop\probleme_competitii_ml\venv_torch\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 5/5 [00:08<00:00,  1.73s/it]


Epoch 2/ 20
Loss: 4.1070 | Val F1: 0.7976190476190477


 80%|████████  | 16/20 [00:47<00:13,  3.28s/it]c:\Users\sabin\Desktop\probleme_competitii_ml\venv_torch\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 5/5 [00:08<00:00,  1.64s/it]


...

Epoch 18/ 20
Loss: 0.3410 | Val F1: 0.8047337278106509


 15%|█▌        | 3/20 [00:11<01:05,  3.84s/it]c:\Users\sabin\Desktop\probleme_competitii_ml\venv_torch\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 5/5 [00:09<00:00,  1.83s/it]


Epoch 19/ 20
Loss: 0.3654 | Val F1: 0.8125


 35%|███▌      | 7/20 [00:14<00:20,  1.54s/it]c:\Users\sabin\Desktop\probleme_competitii_ml\venv_torch\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 5/5 [00:08<00:00,  1.73s/it]

Epoch 20/ 20
Loss: 0.2112 | Val F1: 0.8625


In [38]:
model.eval()

preds = []
paths = []

with torch.no_grad():
    for images, img_paths in test_loader:
        images= images.to(DEVICE)
        outputs = model(images)

        probs= torch.sigmoid(outputs)
        preds_batch = (probs>0.5).int().cpu().numpy().flatten()
        preds.extend(preds_batch)
        paths.extend(img_paths)

In [ ]:
submission = pd.DataFrame({
    'SampleID': ,
    'Label' : preds
})

submission.to_csv('submission_1_mar28th.csv',index = False)